In [0]:
%sql

use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import lit

# Functions

In [0]:
def save_to_parquet(df, save_path,mode):
    (df.write.format('parquet')
        .mode(mode)
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

# Variables

In [0]:
mall_cellsite_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/proj_3/raw/cellsite_chula.parquet'
# mall_cellsite_path = 'gs://tdg-ds-tech-delivery/2026/chula/master/cellsite_chula.parquet'
start_hour = 10
end_hour = 23

# 5-km rectangle boundary -- validated
min_lat = 13.598036 
min_lon = 100.363694
max_lat = 13.924526
max_lon = 100.729024

# BMR
province_bmr = ['Bangkok','Nakhon Pathom','Nonthaburi','Samut Prakan','Samut Sakhon','Pathum Thani']
province_work_home_bmr = ['BANGKOK','SAMUTSAKHON','SAMUTPRAKAN','NAKHONPATHOM','NONTHABURI','PATHUMTHANI']

# # debug 
# start_day = '20260121'
# end_day = '20260131'

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202505")
par_month = dbutils.widgets.get("par_month")

dbutils.widgets.text("start_day", "20250501")
start_day = dbutils.widgets.get("start_day")

dbutils.widgets.text("end_day", "20250510")
end_day = dbutils.widgets.get("end_day")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# customer360 date
par_month_obj = datetime.strptime(str(par_month), '%Y%m')
next_par_month_obj = par_month_obj + relativedelta(months=1)
cust360_date = int(next_par_month_obj.strftime('%Y%m') + '01')

# debug
display('par_month = ',par_month)
display(' customer360_date = ',cust360_date)

# Data Preprocessing

In [0]:
df_cellsite_sk_mall = spark.read.parquet(mall_cellsite_path).select('name','cellsite_sk')
dim_cell = spark.read.table('trueanalytics_data.trueanalytics_base.acsappo_dim_cellsite_location')
df_cellsite_sk_mall_lat_long = df_cellsite_sk_mall\
    .join(dim_cell.select(['cellsite_sk','latitude','longitude']), 'cellsite_sk', 'left')\
    .withColumn('latitude_re', F.round(F.col('latitude'), 3))\
    .withColumn('longitude_re', F.round(F.col('longitude'), 3))

In [0]:
df_cdr_filtered  = spark.read.table('trueanalytics_data.trueanalytics_bus.fact_cdr_geo_agg_hour_v2')\
    .filter(F.col('par_hour').between(start_hour, end_hour))\
    .filter(F.col('par_day').between(start_day, end_day))\
    .filter(F.col('user_estimated_lat')<max_lat).filter(F.col('user_estimated_lat')>min_lat)\
    .filter(F.col('user_estimated_long')<max_lon).filter(F.col('user_estimated_long')>min_lon)

df_cdr_filtered = df_cdr_filtered.select('msisdn','cellsite_sk','total_duration','user_estimated_lat','user_estimated_long','par_day','par_hour')

df_dwell_time = df_cdr_filtered.join(F.broadcast(df_cellsite_sk_mall_lat_long), 'cellsite_sk', 'inner')

In [0]:
max_duration_df = df_dwell_time.groupBy('msisdn','name','latitude_re', 'longitude_re', 'par_hour','par_day')\
    .agg(F.max('total_duration').alias('max_total_duration'))


sum_max_duration_df = max_duration_df.groupBy('msisdn','name', 'par_hour','par_day')\
    .agg(F.sum('max_total_duration').alias('sum_max_total_duration'))\
    .withColumn('actual_total_duration_hr', F.when(F.col('sum_max_total_duration') >3600, lit(3600))\
                .otherwise(F.col('sum_max_total_duration')))

In [0]:
df_mall_time_day = sum_max_duration_df.groupBy('msisdn','name','par_day')\
    .agg(F.sum('actual_total_duration_hr').alias('actual_total_duration_day'))

df_mall_time = sum_max_duration_df.join(df_mall_time_day, on=['msisdn','name','par_day'], how='inner')\
    .select("msisdn","name","par_day","par_hour","actual_total_duration_hr","actual_total_duration_day")

# debug
# display(df_mall_time.count())

# Customer360

In [0]:
columns = ['msisdn','demo_tourist_sim_v1_tourist_bin','geog_resident_location_v1_province_en_cat','geog_resident_location_v1_sub_district_en_cat','geog_resident_location_v1_district_en_cat','geog_work_location_v1_province_en_cat','geog_work_location_v1_sub_district_en_cat','geog_work_location_v1_district_en_cat']
df_cust360 = spark.read.table('trueanalytics_data.customer360.customer360_snapshot')\
    .filter((F.col('par_day')==cust360_date) & (F.col('activated_flag')=='1'))\
    .select(columns)

In [0]:
df_mall_cust360 = df_mall_time.join(df_cust360, on='msisdn', how='left')

# Roaming

In [0]:
save_path_roaming = f'dbfs:/Volumes/idp_int_insight_poc/stage/master/{par_month}_foriegner.parquet'
df_roaming = spark.read.format('parquet').load(save_path_roaming)\
    .filter(F.col('a_country_name')!='thailand')\
        .withColumn('roaming_flag',lit(1))\
        .select('msisdn','a_country_name','roaming_flag')

In [0]:
df_mall_cust360_roaming = df_mall_cust360.join(df_roaming, on='msisdn', how='left')

# Define type of customer
## local_bmr, local_non_bmr, roaming, foriegner

In [0]:
df_format = df_mall_cust360_roaming.withColumn('is_foriegner', F.when( 
    (F.col('demo_tourist_sim_v1_tourist_bin') == 'y') |
    (F.col('roaming_flag')==1), F.lit(1))\
    .otherwise(F.lit(0)))

In [0]:
df_format = df_format.withColumn('is_bmr',
    F.when(
        (F.col('geog_resident_location_v1_province_en_cat').isin(province_work_home_bmr) |
         F.col('geog_work_location_v1_province_en_cat').isin(province_work_home_bmr)) &
        (F.col('demo_tourist_sim_v1_tourist_bin').isNull()) &
        (F.col('roaming_flag').isNull()), F.lit(1)
    ).otherwise(F.lit(0))
)

df_format = df_format.withColumn('is_non_bmr',
    F.when(
        ~(F.col('geog_resident_location_v1_province_en_cat').isin(province_work_home_bmr)) &
        ~(F.col('geog_work_location_v1_province_en_cat').isin(province_work_home_bmr)) &
        (F.col('demo_tourist_sim_v1_tourist_bin').isNull()) &
        (F.col('roaming_flag').isNull()), F.lit(1)
    ).otherwise(F.lit(0))
)

In [0]:
display(df_format.filter((F.col("is_foriegner") + F.col("is_bmr") + F.col("is_non_bmr")) > 1))

# Save the well-preprocessed dataset

In [0]:
save_to_parquet(df_format, f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_footfall.parquet','append')